In [ ]:
import torch

from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM
from vllm.sampling_params import SamplingParams

engine_args = AsyncEngineArgs(
    model="./fastconformer_model/",
    max_model_len=4096,
    gpu_memory_utilization=0.85,
    block_size=128,
    skip_tokenizer_init=True,
    enable_prefix_caching=False,
    dtype="float32",
    compilation_config={"cudagraph_mode": "FULL"},
    enforce_eager=True,
    #load_format="dummy",
)
engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(max_tokens=4096, skip_sampling=True)


# inputs are created as:
# orig_model = EncDecHybridRNNTCTCBPEModel.restore_from(nemo_path, strict=False)
# mel, mel_len = orig_model.preprocessor(input_signal=audio, length=audio_len)
# mel = torch.transpose(mel, 1, 2)  # [B, T, F]
# preprocessed, _ = orig_model.encoder.pre_encode(x=mel, lengths=mel_len)
preprocessed = torch.load("fastconformer_test_data/inputs.pt").cpu().contiguous()
print(preprocessed.shape)
# nemo_enc = ConformerEncoder(...)
# nemo_enc.load_state_dict(...)
# expected_emb, _ = nemo_enc(audio_signal=preprocessed, length=length, bypass_pre_encode=True)
expected_emb = torch.load("fastconformer_test_data/nemo_outputs.pt").cpu().contiguous()
print(expected_emb.shape)


prompt_len = 1
inputs = {
    "prompt_token_ids": [0] * prompt_len,
    "custom_inputs": {
        "proc_melspec": preprocessed[:prompt_len]
    }
}

request_id="1"
i = prompt_len - 1
all_outputs = []
async for output in engine.generate(inputs, sampling_params=sampling_params, request_id=request_id):
    acoustic_emb = output.outputs[0].custom_outputs["acoustic_emb"].clone()
    all_outputs.append(acoustic_emb)

    # prepare of next one
    i += 1
    if i == preprocessed.shape[0]:
        break
    new_input = {"proc_melspec": preprocessed[i:(i+1), :]}
    await engine.append_request(request_id=request_id, custom_inputs=new_input)

pred = torch.cat(all_outputs, dim=0)


In [ ]:
import matplotlib.pyplot as plt

print(expected_emb.shape)
plt.imshow(expected_emb.T, aspect="auto")
plt.colorbar()
plt.show()

print(pred.shape)
plt.imshow(pred.T, aspect="auto")
plt.colorbar()
plt.show()

plt.imshow(pred.T - expected_emb.T, aspect="auto")
plt.colorbar()
plt.show()

